In [1]:
import pandas as pd; df = pd.read_csv('marketing_AB.csv'); print(df.columns)

Index(['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads',
       'most ads day', 'most ads hour'],
      dtype='object')


In [2]:
conversion_summary = df.groupby('test group')['converted'].agg(['count', 'sum', 'mean'])
conversion_summary.columns = ['total_users', 'converted_users', 'conversion_rate']
print(conversion_summary)

            total_users  converted_users  conversion_rate
test group                                               
ad               564577            14423         0.025547
psa               23524              420         0.017854


In [3]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['test group'], df['converted'])
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(f"\nChi-square statistic: {chi2:.2f}")
print(f"p-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print("Statistically significant difference in conversion rates (reject H0)")
else:
    print("No statistically significant difference (fail to reject H0)")

# Lift calculation
ad_rate = conversion_summary.loc['ad', 'conversion_rate']
psa_rate = conversion_summary.loc['psa', 'conversion_rate']
print(f"\nAbsolute difference: {(ad_rate - psa_rate)*100:.2f} percentage points")
print(f"Relative lift: {((ad_rate - psa_rate) / psa_rate)*100:.1f}%")
print(f"p-value: {p_value:.3e}")

converted    False  True 
test group               
ad          550154  14423
psa          23104    420

Chi-square statistic: 54.01
p-value: 0.0000000000
Statistically significant difference in conversion rates (reject H0)

Absolute difference: 0.77 percentage points
Relative lift: 43.1%
p-value: 1.999e-13


In [4]:
import numpy as np

n_ad, x_ad = 550154 + 14423, 14423
n_psa, x_psa = 23104 + 420, 420
p_ad, p_psa = x_ad / n_ad, x_psa / n_psa

diff = p_ad - p_psa
se = np.sqrt(p_ad*(1-p_ad)/n_ad + p_psa*(1-p_psa)/n_psa)
ci_low, ci_high = diff - 1.96*se, diff + 1.96*se
print(f"95% CI for the difference: {ci_low*100:.2f} to {ci_high*100:.2f} percentage points")

95% CI for the difference: 0.60 to 0.94 percentage points


#Findings
Users shown the ad converted at 2.55% versus 1.79% for the PSA control group, a 0.77 percentage-point absolute difference (43% relative lift). A chi-square test of independence found this difference statistically significant (χ² = 54.01, p < 0.001), and the 95% confidence interval for the true difference is roughly 0.60 to 0.94 percentage points.

#Recommendation
The ads drive a real, measurable increase in conversions and the campaign should continue, subject to a cost check: this dataset contains no ad cost or revenue-per-sale figures, so whether a ~0.6–0.9 point lift is profitable depends on those numbers.

#Caveats
The groups are highly unbalanced (~96% ad, ~4% PSA), so the PSA estimate is less precise than the ad estimate. This analysis also assumes users were randomly assigned to groups, as the dataset's documentation states.